# 01 — Residential Mortgage LGD Model

**Australian APRA-Style IRB LGD Framework**

This notebook demonstrates a complete residential mortgage LGD workflow aligned with
Australian Big 4 / APRA practice:

| Step | Description |
|------|-------------|
| 1 | Load synthetic mortgage default & workout data |
| 2 | Exploratory data analysis — LGD distributions, cure rates |
| 3 | Segmentation — Standard/Non-standard × LTV band × LMI |
| 4 | Two-stage model — P(Cure) logistic + LGD\|Loss beta regression |
| 5 | Exposure-weighted long-run LGD |
| 6 | Downturn overlay (macro-linked) |
| 7 | Margin of conservatism |
| 8 | APRA overlays — LMI, floors (10%/15%), standard/non-standard |
| 9 | Illustrative RWA and capital linkage |
| 10 | Validation — accuracy, calibration, PSI, out-of-time backtest |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, classification_report
import statsmodels.api as sm

from src.data_generation import generate_mortgage_data
from src.lgd_calculation import MortgageLGDEngine, exposure_weighted_average
from src.validation import (
    accuracy_metrics, discriminatory_power, conservatism_test,
    calibration_by_segment, compute_psi, out_of_time_split,
    out_of_time_backtest, generate_validation_report
)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** proxy arrears and behavioural drivers are allowed for portfolio demonstration where observed collections history is unavailable.
- **Cure treatment (standard policy):** mortgages use two-stage cure logic `LGD = (1 - cure_rate) * liquidation_loss`.
- **Calibration status (standard policy):** this notebook is a transparent proxy baseline and is **not** internally calibrated to real workout tapes.
- **Product differentiation:** mortgage severity is driven by LVR/LMI and the cure-vs-foreclosure path rather than DSCR/ICR or development completion metrics.
- **Output standard:** report `lgd_base`, `lgd_downturn`, and EAD-weighted outputs for interview-ready comparison.


## Objective
Build an interview-ready residential mortgage LGD module with transparent cure-vs-liquidation logic and exposure-weighted outputs.

## Drivers
- LVR at default
- LMI eligibility and coverage
- Arrears/behavioural proxy inputs
- Cure probability and foreclosure path

## Logic
- Two-stage mortgage treatment: `LGD = (1 - cure_rate) * liquidation_loss`
- Product-appropriate discounting and weighted LGD aggregation

## Downturn
- Macro-linked stress via house-price decline, unemployment, and rate shock channels

## Validation
- EAD-weighted base/downturn outputs, stability checks, and governance flags

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## 1. Generate & Load Data

In [ ]:
loans, cashflows = generate_mortgage_data(n_loans=500, seed=42)

print(f"Loans: {loans.shape}")
print(f"Cashflows: {cashflows.shape}")
print(f"\nDefault date range: {loans['default_date'].min()} to {loans['default_date'].max()}")
print(f"Cure rate: {(loans['resolution_type'] == 'Cure').mean():.1%}")
loans.head()

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics
print("=== Realised LGD Summary ===")
display(loans['realised_lgd'].describe())

print("\n=== By Resolution Type ===")
display(loans.groupby('resolution_type')['realised_lgd'].describe())

In [ ]:
# Bimodal LGD distribution - the key insight driving two-stage modelling
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ewa_realised = exposure_weighted_average(loans, 'realised_lgd', 'ead')
axes[0].hist(loans['realised_lgd'], bins=40, edgecolor='white', alpha=0.8)
axes[0].set_title('All Defaults - Realised LGD')
axes[0].set_xlabel('LGD')
axes[0].axvline(ewa_realised, color='red', ls='--', label=f"EAD-weighted: {ewa_realised:.2%}")
axes[0].legend()

cures = loans[loans['resolution_type'] == 'Cure']['realised_lgd']
losses = loans[loans['resolution_type'] == 'Property Sale']['realised_lgd']

axes[1].hist(cures, bins=30, edgecolor='white', alpha=0.8, color='green')
axes[1].set_title(f'Cures (n={len(cures)}) - Near-Zero LGD')
axes[1].set_xlabel('LGD')

axes[2].hist(losses, bins=30, edgecolor='white', alpha=0.8, color='salmon')
axes[2].set_title(f'Property Sales (n={len(losses)}) - Loss Cases')
axes[2].set_xlabel('LGD')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_lgd_bimodal.png'), dpi=150, bbox_inches='tight')
plt.show()

print()
print('Bimodal distribution confirms need for TWO-STAGE model (cure vs loss).')


In [ ]:
# LTV at default vs LGD — the primary risk driver
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Colour by resolution
for res, colour in [('Cure', 'green'), ('Property Sale', 'salmon')]:
    mask = loans['resolution_type'] == res
    axes[0].scatter(loans.loc[mask, 'ltv_at_default'], loans.loc[mask, 'realised_lgd'],
                    alpha=0.4, s=15, label=res, color=colour)
axes[0].set_xlabel('LTV at Default')
axes[0].set_ylabel('Realised LGD')
axes[0].set_title('LTV at Default vs Realised LGD')
axes[0].legend()

# Boxplot by LTV band
engine = MortgageLGDEngine()
segmented = engine.segment_loans(loans)
segmented.boxplot(column='realised_lgd', by='ltv_band', ax=axes[1])
axes[1].set_title('Realised LGD by LTV Band')
axes[1].set_xlabel('LTV Band')
axes[1].set_ylabel('Realised LGD')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_ltv_vs_lgd.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key driver analysis
print('=== Exposure-Weighted LGD by Key Segments ===')
for col in ['mortgage_class', 'property_type', 'occupancy', 'loan_type', 'state']:
    print()
    print(f"--- {col} ---")
    rows = []
    for seg, grp in loans.groupby(col, observed=True):
        rows.append({
            col: seg,
            'count': len(grp),
            'total_ead': grp['ead'].sum(),
            'cure_rate': (grp['resolution_type'] == 'Cure').mean(),
            'ead_weighted_lgd': exposure_weighted_average(grp, 'realised_lgd', 'ead'),
            'ead_weighted_ltv': exposure_weighted_average(grp, 'ltv_at_default', 'ead'),
        })
    summary = pd.DataFrame(rows).set_index(col).sort_values('ead_weighted_lgd', ascending=False).round(4)
    display(summary)


## 3. Segmentation

Australian mortgage LGD segmentation hierarchy:
```
Level 1: Standard vs Non-Standard  (APRA requirement)
  Level 2: LTV band at default     (primary risk driver)
    Level 3: LMI eligible           (material recovery impact)
```

In [ ]:
# Exposure-weighted long-run LGD by primary segments
engine = MortgageLGDEngine()
lr_lgd = engine.compute_long_run_lgd(loans, segments=['mortgage_class', 'ltv_band'])

print("=== Exposure-Weighted Long-Run LGD ===")
display(lr_lgd)

# Pivot for heatmap
pivot = lr_lgd.pivot(index='mortgage_class', columns='ltv_band', values='lgd_long_run')
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='YlOrRd', ax=ax)
ax.set_title('Exposure-Weighted Long-Run LGD: Mortgage Class × LTV Band')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_segment_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## Cure Treatment (Pre-Step-3 Baseline)

Cure modelling is mandatory in the mortgage baseline because many defaults resolve before liquidation.

- Stage 1: estimate `P(cure)` from approved proxy drivers.
- Stage 2: estimate liquidation loss conditional on non-cure.
- Combine: `LGD = (1 - cure_rate) * liquidation_loss`.

Policy position: fallback and proxy rules follow `docs/fallback_hierarchy.md`; calibration remains proxy-only pending internal data.


In [ ]:
# Prepare modelling data with transparent cure proxies
model_df = loans.copy()
model_df['is_cure'] = (model_df['resolution_type'] == 'Cure').astype(int)

# Borrower type proxy
model_df['borrower_type_proxy'] = np.where(
    model_df['occupancy'] == 'Investor',
    'Investor',
    'Owner-Occupier',
)
model_df['is_investor'] = (model_df['borrower_type_proxy'] == 'Investor').astype(int)

# Arrears stage proxy at default (30-59 / 60-89 / 90+ DPD style buckets)
model_df['arrears_stage_proxy'] = np.select(
    [
        (model_df['ltv_at_default'] >= 0.95) | (model_df['dti'] >= 0.45) | (model_df['credit_score'] < 560),
        (model_df['ltv_at_default'] >= 0.85) | (model_df['dti'] >= 0.35) | (model_df['credit_score'] < 640),
    ],
    ['90+ DPD (Proxy)', '60-89 DPD (Proxy)'],
    default='30-59 DPD (Proxy)',
)
arrears_map = {
    '30-59 DPD (Proxy)': 0,
    '60-89 DPD (Proxy)': 1,
    '90+ DPD (Proxy)': 2,
}
model_df['arrears_stage_num'] = model_df['arrears_stage_proxy'].map(arrears_map).astype(int)

# Repayment behaviour proxy (simple score from borrower risk profile)
repay_score = (
    (model_df['loan_type'] == 'P&I').astype(int)
    + (model_df['credit_score'] >= 700).astype(int)
    + (model_df['seasoning_months'] >= 24).astype(int)
    + (model_df['dti'] <= 0.35).astype(int)
    - (model_df['ltv_at_default'] > 0.90).astype(int)
)
model_df['repayment_behaviour_proxy'] = np.select(
    [repay_score >= 3, repay_score >= 1],
    ['Strong', 'Stable'],
    default='Weak',
)
repayment_map = {'Weak': 0, 'Stable': 1, 'Strong': 2}
model_df['repayment_behaviour_num'] = model_df['repayment_behaviour_proxy'].map(repayment_map).astype(int)

# Additional standard risk flags
model_df['is_non_standard'] = (model_df['mortgage_class'] == 'Non-Standard').astype(int)

features = [
    'ltv_at_default',
    'arrears_stage_num',
    'is_investor',
    'repayment_behaviour_num',
    'is_non_standard',
    'lmi_flag',
]

print('=== Cure Proxy Distributions ===')
display(model_df['arrears_stage_proxy'].value_counts(dropna=False).rename('count').to_frame())
display(model_df['repayment_behaviour_proxy'].value_counts(dropna=False).rename('count').to_frame())


In [ ]:
# STAGE 1: Cure rate model (logistic)
X = model_df[features]
y_cure = model_df['is_cure']

cure_model = LogisticRegression(max_iter=2000, random_state=42)
cure_cv_proba = cross_val_predict(cure_model, X, y_cure, cv=5, method='predict_proba')[:, 1]
cure_model.fit(X, y_cure)

cure_auc = roc_auc_score(y_cure, cure_cv_proba)
print(f"Stage 1 - Cure Model AUC (5-fold CV): {cure_auc:.4f}")

cure_coefs = pd.DataFrame(
    {
        'Feature': features,
        'Coefficient': cure_model.coef_[0],
    }
).sort_values('Coefficient', ascending=False)
print()
print('Cure Model Coefficients:')
display(cure_coefs.round(4))

model_df['cure_rate'] = cure_model.predict_proba(X)[:, 1]


In [ ]:
# STAGE 2: Liquidation loss model (non-cure loans only)
loss_df = model_df[model_df['is_cure'] == 0].copy()
loss_df['liquidation_loss_observed'] = loss_df['realised_lgd']

# Logit transform for bounded target
loss_df['liquidation_loss_clipped'] = loss_df['liquidation_loss_observed'].clip(0.001, 0.999)
loss_df['liquidation_loss_logit'] = np.log(
    loss_df['liquidation_loss_clipped'] / (1 - loss_df['liquidation_loss_clipped'])
)

X_loss = sm.add_constant(loss_df[features])
y_loss = loss_df['liquidation_loss_logit']
loss_model = sm.OLS(y_loss, X_loss).fit()

print('Stage 2 - Liquidation Loss Model (non-cure loans, logit OLS)')
print(f"R-squared: {loss_model.rsquared:.4f}")
print(f"Adj. R-squared: {loss_model.rsquared_adj:.4f}")

loss_coefs = pd.DataFrame(
    {
        'Feature': ['const'] + features,
        'Coefficient': loss_model.params,
        'P-value': loss_model.pvalues,
    }
)
display(loss_coefs.round(4))


In [ ]:
# COMBINE STAGES: LGD = (1 - cure_rate) * liquidation_loss
X_all = sm.add_constant(model_df[features])
liquidation_logit_pred = loss_model.predict(X_all)
model_df['liquidation_loss'] = (1 / (1 + np.exp(-liquidation_logit_pred))).clip(0, 1)

model_df['lgd_predicted'] = (1 - model_df['cure_rate']) * model_df['liquidation_loss']

print('=== Two-Stage Cure Model Output ===')
print(f"EAD-weighted Cure Rate:        {exposure_weighted_average(model_df, 'cure_rate', 'ead'):.2%}")
print(f"EAD-weighted Liquidation Loss: {exposure_weighted_average(model_df, 'liquidation_loss', 'ead'):.2%}")
print(f"EAD-weighted Predicted LGD:    {exposure_weighted_average(model_df, 'lgd_predicted', 'ead'):.2%}")
print(f"EAD-weighted Actual LGD:       {exposure_weighted_average(model_df, 'realised_lgd', 'ead'):.2%}")

cure_segment = (
    model_df.groupby(['arrears_stage_proxy', 'repayment_behaviour_proxy'], observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'count': len(g),
                'total_ead': g['ead'].sum(),
                'ead_weighted_cure_rate': exposure_weighted_average(g, 'cure_rate', 'ead'),
                'ead_weighted_lgd_predicted': exposure_weighted_average(g, 'lgd_predicted', 'ead'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
print()
print('=== Cure Segment Diagnostics ===')
display(cure_segment.round(4))


In [ ]:
# Predicted vs actual scatter
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(model_df['realised_lgd'], model_df['lgd_predicted'], alpha=0.3, s=15)
ax.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Perfect calibration')
ax.set_xlabel('Actual Realised LGD')
ax.set_ylabel('Predicted LGD (Two-Stage)')
ax.set_title('Two-Stage Model: Predicted vs Actual')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_pred_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5–8. Regulatory Pipeline: Long-Run → Downturn → MoC → APRA Overlays

In [ ]:
# Use two-stage cure model LGD as input to regulatory overlays
loans_for_reg = model_df.copy()
loans_for_reg['realised_lgd_actual'] = loans_for_reg['realised_lgd']
loans_for_reg['realised_lgd'] = loans_for_reg['lgd_predicted']

engine = MortgageLGDEngine(downturn_scalar=1.08, moc=0.02)
result = engine.apply_apra_overlays(loans_for_reg)

print('=== Regulatory Pipeline Summary ===')
pipeline_cols = ['lgd_base', 'lgd_downturn', 'lgd_with_moc', 'lgd_after_lmi', 'lgd_final']
display(result[pipeline_cols].describe().round(4))

print()
print('=== Exposure-Weighted Portfolio Averages ===')
for col in pipeline_cols:
    ewa = exposure_weighted_average(result, lgd_col=col, ead_col='ead')
    print(f"  {col:25s}: {ewa:.4%}")

print()
print('=== Downturn Scalar (Mortgage Macro-Linked) ===')
print(
    f"min={result['downturn_scalar'].min():.3f}, "
    f"mean={result['downturn_scalar'].mean():.3f}, "
    f"max={result['downturn_scalar'].max():.3f}"
)


In [ ]:
# Waterfall: Two-Stage LGD -> Downturn -> MoC -> LMI -> Final
pipeline_means = {col: exposure_weighted_average(result, col) for col in pipeline_cols}

fig, ax = plt.subplots(figsize=(10, 5))
labels = ['Base LGD (Cure-Aware)', 'Downturn (Macro)', '+ MoC', 'LMI Adjustment', 'Final (Floor Applied)']
values = [pipeline_means[c] for c in pipeline_cols]
colors = ['#3498db', '#e67e22', '#e74c3c', '#2ecc71', '#8e44ad']

bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{val:.2%}',
        ha='center',
        va='bottom',
        fontweight='bold',
    )

ax.set_ylabel('Exposure-Weighted LGD')
ax.set_title('Mortgage LGD Pipeline with Cure Modelling')
ax.set_ylim(0, max(values) * 1.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_waterfall.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# LGD by mortgage class - Standard vs Non-Standard floor impact
for mc in ['Standard', 'Non-Standard']:
    subset = result[result['mortgage_class'] == mc]
    floor = 0.10 if mc == 'Standard' else 0.15
    floored_pct = ((subset['lgd_after_lmi'] < floor).sum() / len(subset)) if len(subset) else 0.0
    input_ewa = exposure_weighted_average(subset, 'realised_lgd', 'ead')
    final_ewa = exposure_weighted_average(subset, 'lgd_final', 'ead')
    print()
    print(f"{mc} Mortgages (n={len(subset)}):")
    print(f"  EAD-weighted Input LGD:  {input_ewa:.2%}")
    print(f"  EAD-weighted Final LGD:  {final_ewa:.2%}")
    print(f"  Floor ({floor:.0%}) binding: {floored_pct:.1%} of loans")


## 9. Illustrative RWA & Capital

In [ ]:
result_rwa = engine.compute_illustrative_rwa(result, pd_estimate=0.02)

print("=== Illustrative Capital Metrics ===")
print(f"Total EAD:              ${result_rwa['ead'].sum():>15,.0f}")
print(f"Total RWA (pre-APRA):   ${result_rwa['rwa'].sum():>15,.0f}")
print(f"Total RWA (post-APRA):  ${result_rwa['rwa_after_apra_scalar'].sum():>15,.0f}")
print(f"APRA Scalar Impact:     +{(result_rwa['rwa_after_apra_scalar'].sum() / result_rwa['rwa'].sum() - 1):.1%}")
print(f"Average Risk Weight:    {(result_rwa['rwa_after_apra_scalar'].sum() / result_rwa['ead'].sum()):.2%}")

## 10. Validation & Backtesting

In [ ]:
# Full validation report (actual realised LGD vs final model LGD)
segmented_result = engine.segment_loans(result)
val_report = generate_validation_report(
    segmented_result,
    actual_col='realised_lgd_actual',
    predicted_col='lgd_final',
    segment_col='ltv_band',
    date_col='default_date'
)

print('=== Accuracy ===')
for k, v in val_report['accuracy'].items():
    print(f"  {k}: {v}")

print()
print('=== Discriminatory Power ===')
for k, v in val_report['discriminatory_power'].items():
    print(f"  {k}: {v}")

print()
print('=== Conservatism Test ===')
for k, v in val_report['conservatism'].items():
    print(f"  {k}: {v}")


In [ ]:
# Calibration by LTV band
print("=== Calibration by LTV Band ===")
display(val_report.get('calibration', 'No calibration available'))

In [ ]:
# PSI stability
if 'stability_psi' in val_report:
    psi_result = val_report['stability_psi']
    print(f"PSI: {psi_result['PSI']:.4f} — {psi_result['Interpretation']}")
    display(psi_result['Detail'])

In [ ]:
# Out-of-time backtest
if 'out_of_time' in val_report:
    oot = val_report['out_of_time']
    print(f"Train: {oot['train_size']} loans | Test: {oot['test_size']} loans")
    print(f"\nOut-of-Time Accuracy:")
    for k, v in oot['accuracy'].items():
        print(f"  {k}: {v}")
    print(f"\nOut-of-Time Conservatism:")
    for k, v in oot['conservatism'].items():
        print(f"  {k}: {v}")

In [ ]:
# Save outputs
result_rwa.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mortgage_lgd_results.csv'), index=False)
lr_lgd.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mortgage_segment_summary.csv'), index=False)

print("Outputs saved to outputs/tables/")